# LifeMate trên Colab

Chuẩn bị một lần:
1. **Runtime → Change runtime type → GPU L4 hoặc A100.** T4 có thể không chạy được vLLM bản mới.
2. Upload `train+public_test.zip` (có sẵn trong folder project trên Mac) lên Google Drive tại `MyDrive/zalo_lifemate/train+public_test.zip`.
3. Tạo GitHub token (fine-grained, quyền *Contents: Read* cho repo `FirstKhoi/zalo_lifemate`), thêm vào **Colab Secrets** (biểu tượng chìa khoá bên trái) với tên `GH_TOKEN` và bật *Notebook access*.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Code + data

In [ ]:
import os
from google.colab import drive, userdata
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/zalo_lifemate"
REPO = "/content/zalo_lifemate"
token = userdata.get("GH_TOKEN")
if not os.path.isdir(REPO):
    !git clone -q https://{token}@github.com/FirstKhoi/zalo_lifemate.git {REPO}
    !git -C {REPO} remote set-url origin https://github.com/FirstKhoi/zalo_lifemate.git
%cd {REPO}
!git pull -q https://{token}@github.com/FirstKhoi/zalo_lifemate.git main && git log --oneline -1
if not os.path.isdir("train+public_test"):
    !unzip -q "{DRIVE}/train+public_test.zip" -d .
!ls train+public_test

In [ ]:
!pip install -q vllm

In [ ]:
!python test_score.py && python test_agent.py

## 2. Baseline text-only trên dev (Qwen3-4B-Instruct-2507, vLLM)

In [ ]:
for split in ["dev_seen", "dev_unseen"]:
    !python agent.py --ids splits/{split}.txt --out runs/base_{split}.jsonl 2>&1 | grep -E "^\[|Error|Traceback"
    !python score.py runs/base_{split}.jsonl

In [ ]:
!mkdir -p "{DRIVE}/runs" && cp runs/*.jsonl "{DRIVE}/runs/" && ls "{DRIVE}/runs"